In [1]:
# ============================================================
# CONFIGURATION GENERALE
# ============================================================
# Cette cellule initialise l environnement de travail :
# - Import des librairies Python necessaires
# - Definition des chemins vers les dossiers de donnees
# - Definition des departements bretons a filtrer
# - Definition des mappings candidats vers groupes politiques
#
# Les mappings sont des dictionnaires Python qui associent
# le nom de chaque candidat a son groupe politique.
# Ils sont utilises dans la fonction calculer_scores
# pour regrouper les scores par famille politique.
# ============================================================

import pandas as pd   # manipulation de tableaux de donnees
import numpy as np    # calculs mathematiques et statistiques
import os             # gestion des fichiers et dossiers

# Chemins des dossiers de donnees
RAW       = "../data/raw/"        # donnees brutes sources
PROCESSED = "../data/processed/"  # donnees nettoyees et transformees
os.makedirs(PROCESSED, exist_ok=True)  # creer le dossier si inexistant

# Les 4 departements bretons a filtrer
# On utilise des strings car les codes departement
# sont stockes comme du texte dans les fichiers INSEE
DEPTS_BRETAGNE = ['22', '29', '35', '56']

# Mapping des candidats vers les groupes politiques 2017
# Chaque candidat est associe a l un des 5 groupes :
# Ext_gauche, Gauche, Centre, Droite, Ext_droite
# On met les deux versions avec et sans accent
# pour gerer les differences d encodage entre fichiers
MAPPING_2017 = {
    'ARTHAUD'       : 'Ext_gauche',
    'POUTOU'        : 'Ext_gauche',
    'MELENCHON'     : 'Gauche',
    'MÉLENCHON'     : 'Gauche',
    'HAMON'         : 'Gauche',
    'JADOT'         : 'Gauche',
    'MACRON'        : 'Centre',
    'FILLON'        : 'Droite',
    'LE PEN'        : 'Ext_droite',
    'DUPONT-AIGNAN' : 'Ext_droite',
    'LASSALLE'      : 'Ext_droite',
    'ASSELINEAU'    : 'Ext_droite',
    'CHEMINADE'     : 'Ext_droite',
}

# Mapping des candidats vers les groupes politiques 2022
# Les candidats changent entre 2017 et 2022
# mais les 5 groupes politiques restent les memes
# ce qui permet de comparer les deux elections
MAPPING_2022 = {
    'ARTHAUD'       : 'Ext_gauche',
    'POUTOU'        : 'Ext_gauche',
    'MELENCHON'     : 'Gauche',
    'MÉLENCHON'     : 'Gauche',
    'HIDALGO'       : 'Gauche',
    'JADOT'         : 'Gauche',
    'ROUSSEL'       : 'Gauche',
    'MACRON'        : 'Centre',
    'PECRESSE'      : 'Droite',
    'PÉCRESSE'      : 'Droite',
    'LE PEN'        : 'Ext_droite',
    'DUPONT-AIGNAN' : 'Ext_droite',
    'ZEMMOUR'       : 'Ext_droite',
    'LASSALLE'      : 'Ext_droite',
}

print("Configuration OK !")
print(f"Dossier raw       : {os.path.abspath(RAW)}")
print(f"Dossier processed : {os.path.abspath(PROCESSED)}")

Configuration OK !
Dossier raw       : d:\MSPR1\projet\electio-analytics-bretagne\data\raw
Dossier processed : d:\MSPR1\projet\electio-analytics-bretagne\data\processed


In [2]:
# ============================================================
# CHARGEMENT FICHIERS ELECTORAUX
# ============================================================
# On charge les resultats de la presidentielle 2017 et 2022
# Ces fichiers contiennent les scores de chaque candidat
# par commune pour toute la France entiere.
# 
# FICHIER 2017 :
# header=3 car les 3 premieres lignes sont des titres
# parasites avant les vraies colonnes.
# Les colonnes candidats repetees sont nommees
# Nom, Nom.1, Nom.2... car Pandas gere automatiquement
# les noms de colonnes dupliques.
#
# FICHIER 2022 :
# header=0 car les colonnes sont en premiere ligne.
# PROBLEME : les colonnes candidats repetees sont nommees
# Unnamed:26, Unnamed:27... au lieu de Nom.1, Nom.2...
# SOLUTION : on detecte le pattern de base des 7 colonnes
# candidat (N°Panneau, Sexe, Nom, Prenom, Voix,
# %Voix/Ins, %Voix/Exp) et on renomme les Unnamed
# en Nom.1, Nom.2... pour que calculer_scores fonctionne.
# ============================================================

# Chargement fichier electoral 2017
df_2017 = pd.read_excel(
    RAW + "Presidentielle_2017_Resultats_Communes_Tour_1_c.xls",
    header=3  # sauter les 3 lignes de titres parasites
)
print("2017 charge :", df_2017.shape)

# Chargement fichier electoral 2022
df_2022 = pd.read_excel(
    RAW + "resultats-par-niveau-subcom-t1-france-entiere.xlsx",
    header=0  # les colonnes sont bien en premiere ligne
)

# Correction des noms de colonnes Unnamed en Nom.1, Nom.2...
# On detecte le pattern de 7 colonnes par candidat
# a partir de la colonne 19 (premiere colonne candidat)
cols         = df_2022.columns.tolist()
pattern      = cols[19:26]  # 7 colonnes : N°Panneau, Sexe, Nom, Prenom, Voix, %Voix/Ins, %Voix/Exp
nb_candidats = (len(cols) - 19) // len(pattern)

# Renommer les colonnes Unnamed avec le bon nom
nouveaux_noms = cols.copy()
for candidat in range(1, nb_candidats):
    for j, nom in enumerate(pattern):
        idx = 19 + candidat * len(pattern) + j
        if idx < len(cols):
            nouveaux_noms[idx] = f"{nom}.{candidat}"
df_2022.columns = nouveaux_noms

print("2022 charge :", df_2022.shape)
print("Colonnes Nom :", [c for c in df_2022.columns if str(c).startswith('Nom')])

2017 charge : (35719, 95)
2022 charge : (35245, 103)
Colonnes Nom : ['Nom', 'Nom.1', 'Nom.2', 'Nom.3', 'Nom.4', 'Nom.5', 'Nom.6', 'Nom.7', 'Nom.8', 'Nom.9', 'Nom.10', 'Nom.11']


In [3]:
# ============================================================
# FILTRAGE COMMUNES BRETONNES
# ============================================================
# Les fichiers electoraux contiennent toutes les communes
# de France soit environ 35 000 communes.
# On garde uniquement les communes des 4 departements
# bretons : 22 Cotes-d-Armor, 29 Finistere,
# 35 Ille-et-Vilaine, 56 Morbihan.
#
# astype(str).str.strip() : convertit en string et
# supprime les espaces invisibles pour eviter les
# erreurs de comparaison.
#
# isin() : verifie si la valeur est dans la liste
# des departements bretons.
#
# .copy() : cree une copie independante du DataFrame
# pour eviter les avertissements Pandas lors des
# modifications ulterieures.
# ============================================================

# Filtrage communes bretonnes 2017
df_2017['Code du département'] = df_2017['Code du département'].astype(str).str.strip()
bretagne_2017 = df_2017[df_2017['Code du département'].isin(DEPTS_BRETAGNE)].copy()
bretagne_2017['annee'] = 2017

print("2017 - France   :", len(df_2017), "communes")
print("2017 - Bretagne :", len(bretagne_2017), "communes")

# Filtrage communes bretonnes 2022
df_2022['Code du département'] = df_2022['Code du département'].astype(str).str.strip()
bretagne_2022 = df_2022[df_2022['Code du département'].isin(DEPTS_BRETAGNE)].copy()
bretagne_2022['annee'] = 2022

print("2022 - France   :", len(df_2022), "communes")
print("2022 - Bretagne :", len(bretagne_2022), "communes")

2017 - France   : 35719 communes
2017 - Bretagne : 1233 communes
2022 - France   : 35245 communes
2022 - Bretagne : 1207 communes


In [4]:
# ============================================================
# FONCTION CALCUL SCORES PAR GROUPE POLITIQUE
# ============================================================
# Les fichiers electoraux donnent les scores candidat
# par candidat en format large : chaque candidat occupe
# plusieurs colonnes repetees (Nom, Prenom, Voix,
# % Voix/Ins, % Voix/Exp) qui se repetent pour chaque
# candidat sous les noms Nom.1, Nom.2...
#
# Cette fonction prend une ligne du fichier (une commune)
# et retourne les 5 scores de groupes politiques.
#
# La strategie est la suivante :
# 1. Parcourir toutes les colonnes de la ligne
# 2. Pour chaque colonne Nom ou Nom.X trouver le candidat
# 3. Chercher la colonne % Voix/Exp qui suit dans les
#    6 colonnes suivantes
# 4. Appliquer le mapping pour trouver le groupe
# 5. Additionner les scores du meme groupe
#
# float(str(score).replace(',', '.')) : convertit le score
# en nombre en gerant les deux formats decimaux
# virgule francais et point anglais.
# ============================================================

def calculer_scores(row, mapping):
    # Initialiser les scores a zero pour chaque groupe politique
    scores = {
        'score_ext_gauche' : 0.0,
        'score_gauche'     : 0.0,
        'score_centre'     : 0.0,
        'score_droite'     : 0.0,
        'score_ext_droite' : 0.0,
    }

    colonnes = row.index.tolist()

    # Parcourir toutes les colonnes pour trouver les colonnes Nom
    for i, col in enumerate(colonnes):
        # Chercher Nom (premier candidat) et Nom.1, Nom.2... (candidats suivants)
        if col == 'Nom' or str(col).startswith('Nom.'):
            nom_candidat = str(row[col]).strip().upper()

            # Chercher la colonne % Voix/Exp dans les 6 colonnes suivantes
            for j in range(i, min(i + 6, len(colonnes))):
                if '% Voix/Exp' in str(colonnes[j]):
                    score = row[colonnes[j]]

                    # Verifier que le score est un nombre valide
                    if pd.notna(score) and str(score) != 'nan':
                        try:
                            # Convertir en float en gerant la virgule decimale
                            score = float(str(score).replace(',', '.'))
                            # Chercher le groupe du candidat dans le mapping
                            for cle, groupe in mapping.items():
                                if cle in nom_candidat:
                                    scores['score_' + groupe.lower()] += score
                        except ValueError:
                            pass
                    break

    return pd.Series(scores)

print("Fonction calculer_scores OK")

Fonction calculer_scores OK


In [5]:
# ============================================================
# APPLICATION SCORES + FUSION + GAGNANT
# ============================================================
# On applique la fonction calculer_scores sur toutes
# les communes bretonnes pour 2017 et 2022.
#
# On construit ensuite les DataFrames finaux avec
# les colonnes utiles et les scores par groupe.
#
# On fusionne 2017 et 2022 en un seul dataset de 2440 lignes.
#
# Le groupe gagnant est calcule avec idxmax(axis=1) qui
# trouve le nom de la colonne avec la valeur maximale
# sur chaque ligne. On enleve le prefixe score_ pour
# garder uniquement le nom du groupe.
#
# code_insee = code_dept (2 chiffres) + code_commune (3 chiffres)
# Exemple : 22 + 001 = 22001 pour Allineuc
# ============================================================

# Traitement 2017 — application des scores
scores_2017    = bretagne_2017.apply(
    lambda row: calculer_scores(row, MAPPING_2017), axis=1
)

# Construction du DataFrame 2017 avec colonnes utiles
electoral_2017 = pd.concat([
    pd.DataFrame({
        'code_commune'   : bretagne_2017['Code de la commune'].astype(str).str.zfill(3),
        'nom_commune'    : bretagne_2017['Libellé de la commune'],
        'code_dept'      : bretagne_2017['Code du département'],
        'annee'          : 2017,
        'nb_inscrits'    : bretagne_2017['Inscrits'],
        'nb_votants'     : bretagne_2017['Votants'],
        'taux_abstention': bretagne_2017['% Abs/Ins'],
    }).reset_index(drop=True),
    scores_2017.reset_index(drop=True)
], axis=1)
print("2017 traite :", electoral_2017.shape)

# Traitement 2022 — application des scores
scores_2022    = bretagne_2022.apply(
    lambda row: calculer_scores(row, MAPPING_2022), axis=1
)

# Construction du DataFrame 2022 avec colonnes utiles
electoral_2022 = pd.concat([
    pd.DataFrame({
        'code_commune'   : bretagne_2022['Code de la commune'].astype(str).str.zfill(3),
        'nom_commune'    : bretagne_2022['Libellé de la commune'],
        'code_dept'      : bretagne_2022['Code du département'],
        'annee'          : 2022,
        'nb_inscrits'    : bretagne_2022['Inscrits'],
        'nb_votants'     : bretagne_2022['Votants'],
        'taux_abstention': bretagne_2022['% Abs/Ins'],
    }).reset_index(drop=True),
    scores_2022.reset_index(drop=True)
], axis=1)
print("2022 traite :", electoral_2022.shape)

# Fusion 2017 et 2022 en un seul dataset
electoral = pd.concat([electoral_2017, electoral_2022], axis=0).reset_index(drop=True)

# Calcul du groupe gagnant par commune et par annee
# idxmax trouve la colonne avec le score maximum
# str.replace enleve le prefixe score_ du nom
cols_scores = [
    'score_ext_gauche', 'score_gauche', 'score_centre',
    'score_droite', 'score_ext_droite'
]
electoral['gagnant'] = (
    electoral[cols_scores]
    .idxmax(axis=1)
    .str.replace('score_', '', regex=False)
)

# Construction du code INSEE complet sur 5 caracteres
# code_dept sur 2 chiffres + code_commune sur 3 chiffres
electoral['code_insee'] = (
    electoral['code_dept'].astype(str).str.zfill(2) +
    electoral['code_commune'].astype(str).str.zfill(3)
)

print("Dataset electoral :", electoral.shape)
print()
print("Distribution des gagnants :")
print(electoral['gagnant'].value_counts())

2017 traite : (1233, 12)
2022 traite : (1207, 12)
Dataset electoral : (2440, 14)

Distribution des gagnants :
gagnant
ext_droite    959
centre        712
gauche        707
droite         62
Name: count, dtype: int64


In [6]:
# ============================================================
# CHARGEMENT ET EXTRACTION POPULATION
# ============================================================
# On charge les fichiers de population municipale
# pour 2016 et 2021.
#
# POPULATION 2016 :
# Fichier Excel converti en CSV avec 7 lignes d en-tete
# parasites avant les vraies colonnes.
# header=7 permet de sauter ces lignes parasites.
# Le code INSEE est construit en concatenant le code
# departement (2 chiffres) et le code commune (3 chiffres).
# Les nombres contiennent des espaces (ex : 14 081)
# qu on supprime avec str.replace avant conversion.
#
# POPULATION 2021 :
# Fichier CSV direct avec colonnes DEP et CODCOM
# separees qu on concatene pour former le code INSEE.
# ============================================================

# Population 2016
df_pop_2016 = pd.read_csv(
    RAW + "populations_legales_2016.csv",
    sep=";",
    encoding="latin1",  # ← changer utf-8 en latin1
    header=7,
    low_memory=False
)
print("Population 2016 charge :", df_pop_2016.shape)
print("Colonnes :", df_pop_2016.columns.tolist()[:10])

# Construire le code INSEE = code_dept + code_commune
df_pop_2016.columns = df_pop_2016.columns.str.strip()
df_pop_2016['code_dept']    = df_pop_2016.iloc[:, 2].astype(str).str.zfill(2)
df_pop_2016['code_commune'] = df_pop_2016.iloc[:, 5].astype(str).str.zfill(3)
df_pop_2016['code_insee']   = df_pop_2016['code_dept'] + df_pop_2016['code_commune']

# Nettoyer les espaces dans les nombres de population
# ex : 14 081 → 14081
df_pop_2016['PMUN'] = (
    df_pop_2016.iloc[:, 7]
    .astype(str)
    .str.replace(' ', '')
    .str.replace('\xa0', '')  # espace insecable
)
df_pop_2016['PMUN'] = pd.to_numeric(df_pop_2016['PMUN'], errors='coerce')

# Filtrage Bretagne 2016
pop_2016 = df_pop_2016[
    df_pop_2016['code_dept'].isin(DEPTS_BRETAGNE)
][['code_insee', 'PMUN']].copy()
pop_2016.columns = ['code_insee', 'population']
pop_2016['annee'] = 2016

print("Population 2016 Bretagne :", len(pop_2016), "communes")
print(pop_2016.head(3))
print()

# Population 2021
df_pop_2021 = pd.read_csv(
    RAW + "donnees_communes.csv",
    sep=";",
    encoding="latin1",
    low_memory=False
)

# Construire le code INSEE = DEP (2 chiffres) + CODCOM (3 chiffres)
df_pop_2021['DEPCOM'] = (
    df_pop_2021['DEP'].astype(str).str.zfill(2) +
    df_pop_2021['CODCOM'].astype(str).str.zfill(3)
)
df_pop_2021['dept'] = df_pop_2021['DEP'].astype(str).str.zfill(2)

# Filtrage Bretagne 2021
pop_2021 = df_pop_2021[
    df_pop_2021['dept'].isin(DEPTS_BRETAGNE)
][['DEPCOM', 'PMUN']].copy()
pop_2021.columns = ['code_insee', 'population']
pop_2021['annee'] = 2021

print("Population 2021 Bretagne :", len(pop_2021), "communes")
print(pop_2021.head(3))

Population 2016 charge : (35382, 11)
Colonnes : ['Code région', 'Nom de la région', 'Code département', 'Code arrondissement', 'Code canton', 'Code commune', 'Nom de la commune', 'Population municipale', 'Population comptée à part', 'Population totale']
Population 2016 Bretagne : 1232 communes
     code_insee  population  annee
7728      22001         588   2016
7729      22002        1114   2016
7730      22003         974   2016

Population 2021 Bretagne : 1207 communes
     code_insee  population  annee
7287      22001         596   2021
7288      22002        1156   2021
7289      22003         929   2021


In [7]:
# ============================================================
# CHARGEMENT ET EXTRACTION GEO BRETAGNE
# ============================================================
# On charge le fichier communes-france-2022.csv qui contient
# la superficie en km2 et la densite de population
# pour chaque commune de France.
#
# Ces donnees geographiques sont fixes dans le temps :
# la superficie d une commune ne change pas entre 2016
# et 2021. On les utilise donc pour les deux annees.
#
# str.zfill(2) : complete avec des zeros a gauche
# pour avoir toujours 2 caracteres (ex : 01, 22, 29)
# ============================================================

df_geo = pd.read_csv(
    RAW + "communes-france-2022.csv",
    sep=",",
    encoding="utf-8",
    low_memory=False
)

# Formater le code departement sur 2 chiffres
df_geo['dep_code'] = df_geo['dep_code'].astype(str).str.zfill(2)

# Filtrer les communes bretonnes et garder les colonnes utiles
geo_bretagne = df_geo[
    df_geo['dep_code'].isin(DEPTS_BRETAGNE)
][['code_insee', 'superficie_km2', 'densite']].copy()

# S assurer que le code INSEE est sur 5 caracteres
geo_bretagne['code_insee'] = geo_bretagne['code_insee'].astype(str).str.zfill(5)

print("Geo Bretagne :", len(geo_bretagne), "communes")
print(geo_bretagne.head(3))

# ============================================================
# CHARGEMENT ET EXTRACTION VIE ASSOCIATIVE (RNA)
# ============================================================
# On charge le Repertoire National des Associations (RNA)
# pour mesurer la densite associative de chaque commune.
#
# Cette donnee est traitee comme statique (une seule mesure
# recente) car la densite associative d une commune evolue
# tres lentement dans le temps, contrairement au chomage
# ou aux resultats electoraux. On l applique donc de la
# meme facon pour 2016 et 2021.
#
# Traitement :
# 1. Filtrer sur les 4 departements bretons (dep_code)
# 2. Garder uniquement les associations actives
#    (position == 'Active'), on exclut les dissoutes
#    et supprimees
# 3. Compter le nombre d associations par commune
#    (com_code_asso)
# 4. Normaliser par la population 2021 pour obtenir
#    un ratio comparable entre communes :
#    nb_associations_pour_mille = nb_associations / population * 1000
# ============================================================

df_rna = pd.read_csv(
    RAW + "export_rna.csv",
    sep=";",
    encoding="utf-8",
    low_memory=False
)

# Filtrage Bretagne
rna_bretagne = df_rna[
    df_rna['dep_code'].astype(str).isin(DEPTS_BRETAGNE)
].copy()

# Garder uniquement les associations actives
rna_actives = rna_bretagne[rna_bretagne['position'] == 'Active'].copy()

# Formater le code commune sur 5 caracteres
rna_actives['code_insee'] = rna_actives['com_code_asso'].astype(str).str.zfill(5)

# Compter le nombre d associations par commune
nb_assos = rna_actives.groupby('code_insee').size().reset_index(name='nb_associations')

# Normalisation par la population 2021 (donnee de reference recente)
asso_bretagne = nb_assos.merge(
    pop_2021[['code_insee', 'population']], on='code_insee', how='left'
)
asso_bretagne['nb_associations_pour_mille'] = np.where(
    asso_bretagne['population'] > 0,
    (asso_bretagne['nb_associations'] / asso_bretagne['population']) * 1000,
    0
)
asso_bretagne = asso_bretagne[['code_insee', 'nb_associations_pour_mille']]

print("Vie associative Bretagne :", len(asso_bretagne), "communes")
print(asso_bretagne.head(3))

Geo Bretagne : 1208 communes
     code_insee  superficie_km2  densite
7295      22001              24     25.1
7296      22002              12     92.1
7297      22003               6    147.8
Vie associative Bretagne : 1202 communes
  code_insee  nb_associations_pour_mille
0      22001                   33.557047
1      22002                   17.301038
2      22003                   21.528525


In [10]:
# ============================================================
# CHARGEMENT ET EXTRACTION EMPLOI ET EDUCATION
# ============================================================
# On calcule 6 variables socio-economiques :
#
# CHOMAGE :
# taux_chomage = (chomeurs / actifs) * 100
# 2016 : P16_CHOM1564 / P16_ACT1564 depuis fichier emploi 2016
# 2021 : P21_CHOM1564 / P21_ACT1564 depuis fichier emploi 2021
#
# EDUCATION (2016 et 2021) :
# Source : base INSEE "Diplomes - Formation" (niveau IRIS,
# feuille "IRIS"), agregee par commune (somme des IRIS
# partageant le meme code COM), pour obtenir une
# nomenclature coherente sur les deux annees :
#
#   diplome_sup  = part de la population non scolarisee de
#                  15 ans ou plus ayant un diplome de
#                  l enseignement superieur (Bac+2 et plus)
#                  2016 : P16_NSCOL15P_SUP
#                  2021 : P21_NSCOL15P_SUP2 + SUP34 + SUP5
#
#   sans_diplome = part de la population non scolarisee de
#                  15 ans ou plus sans diplome ou avec au
#                  plus un BEPC/brevet
#                  2016 : P16_NSCOL15P_DIPLMIN
#                  2021 : P21_NSCOL15P_DIPLMIN + BEPC
#
# np.where protege contre la division par zero :
# si le denominateur est 0 on retourne 0 plutot que NaN
# ============================================================

# --- CHOMAGE 2016 ---
df_emploi_2016 = pd.read_csv(
    RAW + "base-cc-emploi-pop-active-2016.CSV",
    sep=";", encoding="latin1", low_memory=False
)
df_emploi_2016['CODGEO'] = df_emploi_2016['CODGEO'].astype(str).str.zfill(5)
df_emploi_2016['dept']   = df_emploi_2016['CODGEO'].str[:2]
emploi_bret_2016 = df_emploi_2016[df_emploi_2016['dept'].isin(DEPTS_BRETAGNE)].copy()
emploi_bret_2016['taux_chomage_2016'] = np.where(
    emploi_bret_2016['P16_ACT1564'] > 0,
    (emploi_bret_2016['P16_CHOM1564'] / emploi_bret_2016['P16_ACT1564']) * 100, 0
)
chomage_2016 = emploi_bret_2016[['CODGEO', 'taux_chomage_2016']].copy()
chomage_2016.columns = ['code_insee', 'taux_chomage_2016']
print("Chomage 2016 Bretagne :", len(chomage_2016), "communes")

# --- CHOMAGE 2021 ---
df_emploi_2021 = pd.read_csv(
    RAW + "base-cc-emploi-pop-active-2021.CSV",
    sep=";", encoding="latin1", low_memory=False
)
df_emploi_2021['CODGEO'] = df_emploi_2021['CODGEO'].astype(str).str.zfill(5)
df_emploi_2021['dept']   = df_emploi_2021['CODGEO'].str[:2]
emploi_bret_2021 = df_emploi_2021[df_emploi_2021['dept'].isin(DEPTS_BRETAGNE)].copy()
emploi_bret_2021['taux_chomage_2021'] = np.where(
    emploi_bret_2021['P21_ACT1564'] > 0,
    (emploi_bret_2021['P21_CHOM1564'] / emploi_bret_2021['P21_ACT1564']) * 100, 0
)
chomage_2021 = emploi_bret_2021[['CODGEO', 'taux_chomage_2021']].copy()
chomage_2021.columns = ['code_insee', 'taux_chomage_2021']
print("Chomage 2021 Bretagne :", len(chomage_2021), "communes")
print()

# --- EDUCATION 2016 ---
dipl_2016 = pd.read_excel(
    RAW + "base-ic-diplomes-formation-2016.xls",
    sheet_name='IRIS',
    header=5,
    usecols=['COM', 'DEP', 'P16_NSCOL15P', 'P16_NSCOL15P_DIPLMIN', 'P16_NSCOL15P_SUP']
)
dipl_2016['DEP'] = dipl_2016['DEP'].astype(str).str.zfill(2)
dipl_2016 = dipl_2016[dipl_2016['DEP'].isin(DEPTS_BRETAGNE)].copy()
dipl_2016['COM'] = dipl_2016['COM'].astype(str).str.zfill(5)

edu_2016 = dipl_2016.groupby('COM')[['P16_NSCOL15P', 'P16_NSCOL15P_DIPLMIN', 'P16_NSCOL15P_SUP']].sum()
edu_2016['pct_sans_diplome_2016'] = edu_2016['P16_NSCOL15P_DIPLMIN'] / edu_2016['P16_NSCOL15P'] * 100
edu_2016['pct_diplome_sup_2016']  = edu_2016['P16_NSCOL15P_SUP']     / edu_2016['P16_NSCOL15P'] * 100
edu_2016 = edu_2016[['pct_sans_diplome_2016', 'pct_diplome_sup_2016']].reset_index()
edu_2016.columns = ['code_insee', 'pct_sans_diplome_2016', 'pct_diplome_sup_2016']
print("Education 2016 Bretagne :", len(edu_2016), "communes")

# --- EDUCATION 2021 ---
dipl_2021 = pd.read_csv(
    RAW + "base-ic-diplomes-formation-2021.csv",
    sep=';',
    usecols=['COM', 'P21_NSCOL15P', 'P21_NSCOL15P_DIPLMIN', 'P21_NSCOL15P_BEPC',
             'P21_NSCOL15P_SUP2', 'P21_NSCOL15P_SUP34', 'P21_NSCOL15P_SUP5'],
    dtype={'COM': str}
)
dipl_2021['COM']  = dipl_2021['COM'].str.zfill(5)
dipl_2021['dept'] = dipl_2021['COM'].str[:2]
dipl_2021 = dipl_2021[dipl_2021['dept'].isin(DEPTS_BRETAGNE)].copy()

edu_2021 = dipl_2021.groupby('COM')[[
    'P21_NSCOL15P', 'P21_NSCOL15P_DIPLMIN', 'P21_NSCOL15P_BEPC',
    'P21_NSCOL15P_SUP2', 'P21_NSCOL15P_SUP34', 'P21_NSCOL15P_SUP5'
]].sum()
edu_2021['pct_sans_diplome_2021'] = (
    (edu_2021['P21_NSCOL15P_DIPLMIN'] + edu_2021['P21_NSCOL15P_BEPC']) / edu_2021['P21_NSCOL15P'] * 100
)
edu_2021['pct_diplome_sup_2021'] = (
    (edu_2021['P21_NSCOL15P_SUP2'] + edu_2021['P21_NSCOL15P_SUP34'] + edu_2021['P21_NSCOL15P_SUP5'])
    / edu_2021['P21_NSCOL15P'] * 100
)
edu_2021 = edu_2021[['pct_sans_diplome_2021', 'pct_diplome_sup_2021']].reset_index()
edu_2021.columns = ['code_insee', 'pct_sans_diplome_2021', 'pct_diplome_sup_2021']
print("Education 2021 Bretagne :", len(edu_2021), "communes")

Chomage 2016 Bretagne : 1208 communes
Chomage 2021 Bretagne : 1206 communes

Education 2016 Bretagne : 1232 communes
Education 2021 Bretagne : 1207 communes


In [11]:
# ============================================================
# CHARGEMENT ET EXTRACTION REVENU MEDIAN
# ============================================================
# Le revenu median est le revenu au-dessus et en-dessous
# duquel se trouvent 50% des habitants.
# C est un meilleur indicateur que la moyenne car il
# n est pas fausse par les tres hauts revenus.
#
# REVENU 2016 :
# Fichier Filosofi 2016 converti en CSV
# header=5 car 5 lignes d en-tete parasites
# MED16 = revenu median 2016
# Valeurs manquantes : secret statistique INSEE
# pour les communes de moins de 500 menages fiscaux
# On impute par la mediane du departement
#
# REVENU 2021 :
# Fichier Filosofi 2021
# Q221 = revenu median 2021
# Problemes specifiques :
# - Valeurs s = secret statistique → remplacer par NaN
# - Separateur decimal virgule → remplacer par point
# On impute les NaN par la mediane du departement
# ============================================================

# --- REVENU MEDIAN 2016 ---
df_rev_2016 = pd.read_csv(
    RAW + "cc_filosofi_2016_COM.csv",
    sep=";",
    encoding="latin1",  
    header=5,
    low_memory=False
)
print("Revenu 2016 charge :", df_rev_2016.shape)
print("Colonnes :", df_rev_2016.columns.tolist()[:5])

# Formater CODGEO sur 5 caracteres
df_rev_2016['CODGEO'] = df_rev_2016['CODGEO'].astype(str).str.zfill(5)
df_rev_2016['dept']   = df_rev_2016['CODGEO'].str[:2]

# Filtrage Bretagne et extraction MED16
rev_2016 = df_rev_2016[
    df_rev_2016['dept'].isin(DEPTS_BRETAGNE)
][['CODGEO', 'MED16']].copy()
rev_2016.columns = ['code_insee', 'revenu_median_2016']

# Convertir en numerique (gere les valeurs non numeriques)
rev_2016['revenu_median_2016'] = pd.to_numeric(
    rev_2016['revenu_median_2016'], errors='coerce'
)

# Imputer les NaN par la mediane du departement
# car une commune sans revenu disponible ressemble
# aux communes voisines du meme departement
rev_2016['dept'] = rev_2016['code_insee'].str[:2]
rev_2016['revenu_median_2016'] = rev_2016.groupby('dept')['revenu_median_2016'].transform(
    lambda x: x.fillna(x.median())
)
rev_2016 = rev_2016[['code_insee', 'revenu_median_2016']]

print("Revenu 2016 Bretagne :", len(rev_2016), "communes")
print("NaN restants :", rev_2016['revenu_median_2016'].isna().sum())
print(rev_2016.head(3))
print()

# --- REVENU MEDIAN 2021 ---
df_rev_2021 = pd.read_csv(
    RAW + "FILO2021_DEC_COM.csv",
    sep=";",
    encoding="latin1",
    low_memory=False
)

df_rev_2021['CODGEO'] = df_rev_2021['CODGEO'].astype(str).str.zfill(5)
df_rev_2021['dept']   = df_rev_2021['CODGEO'].str[:2]

rev_2021 = df_rev_2021[
    df_rev_2021['dept'].isin(DEPTS_BRETAGNE)
][['CODGEO', 'Q221']].copy()
rev_2021.columns = ['code_insee', 'revenu_median_2021']

# Remplacer s (secret statistique) par NaN
rev_2021['revenu_median_2021'] = rev_2021['revenu_median_2021'].replace('s', np.nan)

# Remplacer la virgule decimale par un point
rev_2021['revenu_median_2021'] = (
    rev_2021['revenu_median_2021'].astype(str).str.replace(',', '.')
)
rev_2021['revenu_median_2021'] = pd.to_numeric(
    rev_2021['revenu_median_2021'], errors='coerce'
)

# Imputation par mediane du departement
rev_2021['dept'] = rev_2021['code_insee'].str[:2]
rev_2021['revenu_median_2021'] = rev_2021.groupby('dept')['revenu_median_2021'].transform(
    lambda x: x.fillna(x.median())
)
rev_2021 = rev_2021[['code_insee', 'revenu_median_2021']]

print("Revenu 2021 Bretagne :", len(rev_2021), "communes")
print("NaN restants :", rev_2021['revenu_median_2021'].isna().sum())
print(rev_2021.head(3))

Revenu 2016 charge : (34932, 246)
Colonnes : ['CODGEO', 'LIBGEO', 'NBMENFISC16', 'NBPERSMENFISC16', 'MED16']
Revenu 2016 Bretagne : 1206 communes
NaN restants : 0
     code_insee  revenu_median_2016
7296      22001             19870.0
7297      22002             20962.0
7298      22003             20448.0

Revenu 2021 Bretagne : 1207 communes
NaN restants : 0
     code_insee  revenu_median_2021
7292      22001             21360.0
7293      22002             24680.0
7294      22003             22150.0


In [12]:
# ============================================================
# CHARGEMENT ET EXTRACTION CRIMINALITE
# ============================================================
# On charge le fichier de la base communale de delinquance
# du SSMSI qui contient le nombre d infractions par commune
# par annee et par type d infraction.
#
# Le fichier est en format long :
# plusieurs lignes par commune par annee
# une ligne par type d infraction
#
# Strategie de traitement :
# 1. Filtrer annees 2016 et 2021
# 2. Filtrer communes bretonnes
# 3. Garder uniquement les lignes diffusees
#    est_diffuse = diff signifie que la donnee est disponible
#    est_diffuse = ndiff signifie que la donnee n est pas publiee
# 4. Remplacer NaN dans nombre par 0
#    car absence de donnee = 0 infraction dans les petites communes
# 5. Calculer taux = (nombre infractions / population) * 1000
#    On utilise le taux pour 1000 habitants plutot que le nombre
#    brut pour comparer equitablement les communes de tailles
#    differentes. Une grande ville a mecaniquement plus d infractions
#    qu un village donc le nombre brut n est pas comparable.
# ============================================================

# Trouver le fichier de criminalite dans le dossier raw
fichier_crime = [f for f in os.listdir(RAW) if 'donnee' in f.lower()][0]
print("Fichier criminalite :", fichier_crime)

df_crime = pd.read_csv(
    RAW + fichier_crime,
    sep=";",
    encoding="latin1",
    low_memory=False
)
print("Criminalite charge :", df_crime.shape)

# Filtrage annees 2016 et 2021
df_crime_filtre = df_crime[df_crime['annee'].isin([2016, 2021])].copy()

# Formater le code commune sur 5 caracteres
df_crime_filtre['CODGEO_2025'] = df_crime_filtre['CODGEO_2025'].astype(str).str.zfill(5)
df_crime_filtre['dept'] = df_crime_filtre['CODGEO_2025'].str[:2]

# Filtrage Bretagne
df_crime_bret = df_crime_filtre[df_crime_filtre['dept'].isin(DEPTS_BRETAGNE)].copy()

# Garder uniquement les lignes dont la donnee est diffusee
df_crime_diff = df_crime_bret[df_crime_bret['est_diffuse'] == 'diff'].copy()

# Convertir nombre et population en numerique
# fillna(0) : les NaN dans nombre signifient 0 infraction
df_crime_diff['nombre']    = pd.to_numeric(df_crime_diff['nombre'],    errors='coerce').fillna(0)
df_crime_diff['insee_pop'] = pd.to_numeric(df_crime_diff['insee_pop'], errors='coerce').fillna(0)

# Agreger tous les types d infractions par commune et annee
crime_total = df_crime_diff.groupby(
    ['CODGEO_2025', 'annee', 'insee_pop']
)['nombre'].sum().reset_index()

# Calcul du taux pour 1000 habitants
# np.where protege contre la division par zero
crime_total['taux_criminalite'] = np.where(
    crime_total['insee_pop'] > 0,
    (crime_total['nombre'] / crime_total['insee_pop']) * 1000,
    0
)

crime_final = crime_total[['CODGEO_2025', 'annee', 'taux_criminalite']].copy()
crime_final.columns = ['code_insee', 'annee', 'taux_criminalite']

# Separer 2016 et 2021 pour la jointure finale
crime_2016 = crime_final[crime_final['annee'] == 2016][['code_insee', 'taux_criminalite']].copy()
crime_2016.columns = ['code_insee', 'taux_criminalite_2016']
crime_2021 = crime_final[crime_final['annee'] == 2021][['code_insee', 'taux_criminalite']].copy()
crime_2021.columns = ['code_insee', 'taux_criminalite_2021']

print("Criminalite 2016 Bretagne :", len(crime_2016), "communes")
print("Criminalite 2021 Bretagne :", len(crime_2021), "communes")
print(crime_2016[crime_2016['taux_criminalite_2016'] > 0].head(3))

Fichier criminalite : donnee-data.gouv-2025-geographie2025-produit-le2026-02-03 (1).csv
Criminalite charge : (5238000, 13)
Criminalite 2016 Bretagne : 1202 communes
Criminalite 2021 Bretagne : 1202 communes
   code_insee  taux_criminalite_2016
6       22004              18.223712
12      22008               5.469462
20      22013               6.852248


In [13]:
# ============================================================
# CHARGEMENT ET EXTRACTION ENTREPRISES
# ============================================================
# On charge le fichier INSEE SIDE qui contient le nombre
# d entreprises actives par commune par annee et par secteur.
#
# Filtres appliques :
# GEO_OBJECT = COM : uniquement les communes
#              le fichier contient aussi des donnees
#              par bassin de vie et France entiere
# ACTIVITY = _T   : toutes activites confondues
#              on veut le total pas par secteur
# TIME_PERIOD = 2016 et 2021 : nos deux annees
#
# On calcule un ratio entreprises pour 1000 habitants
# plutot que le nombre brut car une grande ville a
# mecaniquement plus d entreprises qu un village.
# Le ratio permet de comparer equitablement les communes
# de tailles differentes et mesure le dynamisme
# economique independamment de la taille de la commune.
# ============================================================

df_ent = pd.read_csv(
    RAW + "DS_SIDE_STOCKS_ET_COM_2023_data.csv",
    sep=";",
    encoding="latin1",
    low_memory=False
)

# Filtrage communes + toutes activites + annees 2016 et 2021
df_ent_filtre = df_ent[
    (df_ent['GEO_OBJECT'] == 'COM') &
    (df_ent['ACTIVITY'] == '_T') &
    (df_ent['TIME_PERIOD'].isin([2016, 2021]))
].copy()

# Formater le code commune sur 5 caracteres
df_ent_filtre['GEO']  = df_ent_filtre['GEO'].astype(str).str.zfill(5)
df_ent_filtre['dept'] = df_ent_filtre['GEO'].str[:2]

# Filtrage Bretagne
ent_bretagne = df_ent_filtre[df_ent_filtre['dept'].isin(DEPTS_BRETAGNE)].copy()
ent_final = ent_bretagne[['GEO', 'TIME_PERIOD', 'OBS_VALUE']].copy()
ent_final.columns = ['code_insee', 'annee', 'nb_entreprises']

# Fusion avec population pour calculer le ratio
# On utilise pop_2016 pour 2016 et pop_2021 pour 2021
pop_2016_ent        = pop_2016[['code_insee', 'population']].copy()
pop_2016_ent['annee'] = 2016
pop_2021_ent        = pop_2021[['code_insee', 'population']].copy()
pop_2021_ent['annee'] = 2021
pop_concat          = pd.concat([pop_2016_ent, pop_2021_ent], axis=0)

ent_final = ent_final.merge(pop_concat, on=['code_insee', 'annee'], how='left')

# Calcul du ratio entreprises pour 1000 habitants
ent_final['nb_entreprises_pour_mille'] = np.where(
    ent_final['population'] > 0,
    (ent_final['nb_entreprises'] / ent_final['population']) * 1000,
    0
)

# Separer 2016 et 2021 pour la jointure finale
ent_2016 = ent_final[ent_final['annee'] == 2016][['code_insee', 'nb_entreprises_pour_mille']].copy()
ent_2016.columns = ['code_insee', 'nb_entreprises_pour_mille_2016']
ent_2021 = ent_final[ent_final['annee'] == 2021][['code_insee', 'nb_entreprises_pour_mille']].copy()
ent_2021.columns = ['code_insee', 'nb_entreprises_pour_mille_2021']

print("Entreprises 2016 Bretagne :", len(ent_2016), "communes")
print("Entreprises 2021 Bretagne :", len(ent_2021), "communes")
print(ent_2016.head(3))

Entreprises 2016 Bretagne : 1202 communes
Entreprises 2021 Bretagne : 1202 communes
  code_insee  nb_entreprises_pour_mille_2016
3      22036                       19.704433
6      22004                       51.738584
7      22045                       43.835616


In [14]:
# ============================================================
# JOINTURE FINALE
# ============================================================
# On fusionne tous les tableaux en un seul dataset complet
# avec une ligne par commune par annee.
#
# LOGIQUE DE CORRESPONDANCE DES ANNEES :
# Les donnees electorales utilisent les annees 2017 et 2022
# Les indicateurs socio-economiques utilisent 2016 et 2021
# car ce sont les donnees disponibles avant chaque election.
#
# Pour faire la jointure correctement on renomme :
# population 2016 → annee 2017 (pour matcher electoral 2017)
# population 2021 → annee 2022 (pour matcher electoral 2022)
#
# Pour les autres indicateurs (chomage, revenus, education,
# criminalite, entreprises) on a deux colonnes separees
# une par annee donc pas de probleme de correspondance.
#
# La vie associative (asso_bretagne) est statique :
# une seule colonne nb_associations_pour_mille appliquee
# de la meme facon pour 2016 et 2021, comme la superficie
# et la densite.
#
# merge avec how='left' : on garde toutes les communes
# du dataset electoral meme si elles n ont pas de
# correspondance dans les autres tableaux (NaN)
# ============================================================

print("Dataset electoral :", electoral.shape)

# Jointure avec population
# On renomme les annees pour correspondre au dataset electoral
pop_2016_join         = pop_2016.copy()
pop_2016_join['annee'] = 2017  # pop 2016 correspond a election 2017
pop_2022_join         = pop_2021.copy()
pop_2022_join['annee'] = 2022  # pop 2021 correspond a election 2022
pop_all = pd.concat([pop_2016_join, pop_2022_join], axis=0)

dataset = electoral.merge(pop_all, on=['code_insee', 'annee'], how='left')
print("Apres population :", dataset.shape)

# Jointure avec superficie et densite (fixes dans le temps)
dataset = dataset.merge(geo_bretagne, on='code_insee', how='left')
print("Apres geo :", dataset.shape)

# Jointure avec vie associative (fixe dans le temps)
dataset = dataset.merge(asso_bretagne, on='code_insee', how='left')
print("Apres vie associative :", dataset.shape)

# Jointure avec chomage
# Deux colonnes : taux_chomage_2016 et taux_chomage_2021
dataset = dataset.merge(chomage_2016, on='code_insee', how='left')
dataset = dataset.merge(chomage_2021, on='code_insee', how='left')
print("Apres chomage :", dataset.shape)

# Jointure avec education 2016
dataset = dataset.merge(edu_2016, on='code_insee', how='left')
print("Apres education 2016 :", dataset.shape)

# Jointure avec education 2021
dataset = dataset.merge(edu_2021, on='code_insee', how='left')
print("Apres education 2021 :", dataset.shape)

# Jointure avec revenus
# Deux colonnes : revenu_median_2016 et revenu_median_2021
dataset = dataset.merge(rev_2016, on='code_insee', how='left')
dataset = dataset.merge(rev_2021, on='code_insee', how='left')
print("Apres revenus :", dataset.shape)

# Jointure avec criminalite
# Deux colonnes : taux_criminalite_2016 et taux_criminalite_2021
dataset = dataset.merge(crime_2016, on='code_insee', how='left')
dataset = dataset.merge(crime_2021, on='code_insee', how='left')
print("Apres criminalite :", dataset.shape)

# Jointure avec entreprises
# Deux colonnes : nb_entreprises_pour_mille_2016 et 2021
dataset = dataset.merge(ent_2016, on='code_insee', how='left')
dataset = dataset.merge(ent_2021, on='code_insee', how='left')
print("Apres entreprises :", dataset.shape)

print()
print("Colonnes finales :")
print(dataset.columns.tolist())

Dataset electoral : (2440, 14)
Apres population : (2440, 15)
Apres geo : (2440, 17)
Apres vie associative : (2440, 18)
Apres chomage : (2440, 20)
Apres education 2016 : (2440, 22)
Apres education 2021 : (2440, 24)
Apres revenus : (2440, 26)
Apres criminalite : (2440, 28)
Apres entreprises : (2440, 30)

Colonnes finales :
['code_commune', 'nom_commune', 'code_dept', 'annee', 'nb_inscrits', 'nb_votants', 'taux_abstention', 'score_ext_gauche', 'score_gauche', 'score_centre', 'score_droite', 'score_ext_droite', 'gagnant', 'code_insee', 'population', 'superficie_km2', 'densite', 'nb_associations_pour_mille', 'taux_chomage_2016', 'taux_chomage_2021', 'pct_sans_diplome_2016', 'pct_diplome_sup_2016', 'pct_sans_diplome_2021', 'pct_diplome_sup_2021', 'revenu_median_2016', 'revenu_median_2021', 'taux_criminalite_2016', 'taux_criminalite_2021', 'nb_entreprises_pour_mille_2016', 'nb_entreprises_pour_mille_2021']


In [15]:
# ============================================================
# VERIFICATION ET NETTOYAGE DU DATASET
# ============================================================
# On verifie les dimensions les types et les valeurs
# manquantes du dataset final.
#
# Les valeurs manquantes proviennent de trois sources :
# 1. Secret statistique INSEE : communes trop petites
# 2. Fusions de communes : codes INSEE qui ne correspondent
#    plus entre 2016 et 2022
# 3. Donnees criminalite non diffusees pour certaines communes
#
# On impute les valeurs manquantes par la mediane
# du departement car une commune sans donnee disponible
# ressemble probablement aux communes voisines
# du meme departement.
# Cette strategie est plus pertinente que la mediane
# nationale car elle tient compte des specificites
# geographiques et socio-economiques locales.
# ============================================================

print("Dimensions dataset :", dataset.shape)
print()

# Verifier les valeurs manquantes avant imputation
print("Valeurs manquantes par colonne :")
nan_counts = dataset.isnull().sum()
print(nan_counts[nan_counts > 0])
print()

# Liste des colonnes numeriques a imputer
cols_a_imputer = [
    'population',
    'superficie_km2',
    'densite',
    'nb_associations_pour_mille',
    'taux_chomage_2016',
    'taux_chomage_2021',
    'pct_sans_diplome_2016',
    'pct_diplome_sup_2016',
    'pct_sans_diplome_2021',
    'pct_diplome_sup_2021',
    'revenu_median_2016',
    'revenu_median_2021',
    'taux_criminalite_2016',
    'taux_criminalite_2021',
    'nb_entreprises_pour_mille_2016',
    'nb_entreprises_pour_mille_2021'
]

# Imputation par mediane du departement
# groupby('dept') : regrouper par departement
# transform : appliquer la fonction sur chaque groupe
# fillna(x.median()) : remplacer NaN par la mediane du groupe
dataset['dept'] = dataset['code_insee'].str[:2]
for col in cols_a_imputer:
    if col in dataset.columns:
        nb_nan_avant = dataset[col].isna().sum()
        if nb_nan_avant > 0:
            dataset[col] = dataset.groupby('dept')[col].transform(
                lambda x: x.fillna(x.median())
            )
            nb_nan_apres = dataset[col].isna().sum()
            print(f"  {col:40s} : {nb_nan_avant} NaN → {nb_nan_apres} NaN")

# Supprimer la colonne departement ajoutee pour l imputation
dataset = dataset.drop(columns=['dept'])

print()
print("Valeurs manquantes apres imputation :")
nan_apres = dataset.isnull().sum()
nan_restants = nan_apres[nan_apres > 0]
if len(nan_restants) == 0:
    print("Aucune valeur manquante !")
else:
    print(nan_restants)

Dimensions dataset : (2440, 30)

Valeurs manquantes par colonne :
population                         1
superficie_km2                    25
densite                           25
nb_associations_pour_mille        36
taux_chomage_2016                 25
taux_chomage_2021                 28
pct_sans_diplome_2016              1
pct_diplome_sup_2016               1
pct_sans_diplome_2021             26
pct_diplome_sup_2021              26
revenu_median_2016                29
revenu_median_2021                26
taux_criminalite_2016             36
taux_criminalite_2021             36
nb_entreprises_pour_mille_2016    36
nb_entreprises_pour_mille_2021    36
dtype: int64

  population                               : 1 NaN → 0 NaN
  superficie_km2                           : 25 NaN → 0 NaN
  densite                                  : 25 NaN → 0 NaN
  nb_associations_pour_mille               : 36 NaN → 0 NaN
  taux_chomage_2016                        : 25 NaN → 0 NaN
  taux_chomage_2021          

In [16]:
# ============================================================
# CORRECTION TYPES ET SAUVEGARDE DATASET FINAL
# ============================================================
# On corrige d abord les types des colonnes codes
# qui doivent etre en string et non en entier
# pour preserver les zeros devant les codes INSEE.
# Ex : departement 01 ne doit pas devenir 1
# Ce n est pas un probleme pour la Bretagne
# (departements 22, 29, 35, 56) mais c est une
# bonne pratique pour la reproductibilite du projet
# si on veut etendre a d autres regions.
#
# On sauvegarde ensuite le dataset complet en CSV
# dans le dossier processed.
# Ce fichier sera utilise directement par
# le notebook d analyse exploratoire et le notebook ML.
#
# index=False : ne pas sauvegarder l index Pandas
# encoding=utf-8 : encodage universel pour eviter
# les problemes d accents sur differents systemes
#
# Le dataset final contient :
# 2440 lignes (1233 communes 2017 + 1207 communes 2022)
# 30 colonnes dont la variable cible gagnant
# 0 valeur manquante apres imputation
# ============================================================

# Supprimer la colonne dept si elle existe encore
if 'dept' in dataset.columns:
    dataset = dataset.drop(columns=['dept'])

# Correction des types des colonnes codes
# Les codes doivent etre en string pour preserver
# les zeros devant les codes INSEE
dataset['code_commune'] = dataset['code_commune'].astype(str).str.zfill(3)
dataset['code_dept']    = dataset['code_dept'].astype(str).str.zfill(2)
dataset['code_insee']   = dataset['code_insee'].astype(str).str.zfill(5)

print("Types apres correction :")
print(f"  code_commune : {dataset['code_commune'].dtype}")
print(f"  code_dept    : {dataset['code_dept'].dtype}")
print(f"  code_insee   : {dataset['code_insee'].dtype}")
print()
print("Exemples codes :")
print(dataset[['code_commune', 'code_dept', 'code_insee']].head(3))
print()

# Sauvegarde en CSV
dataset.to_csv(
    PROCESSED + "dataset_bretagne.csv",
    index=False,
    encoding='utf-8'
)

print("Dataset sauvegarde !")
print("Fichier : dataset_bretagne.csv")
print("Dimensions :", dataset.shape)
print()
print("Apercu :")
print(dataset.head(3))

Types apres correction :
  code_commune : object
  code_dept    : object
  code_insee   : object

Exemples codes :
  code_commune code_dept code_insee
0          001        22      22001
1          002        22      22002
2          003        22      22003

Dataset sauvegarde !
Fichier : dataset_bretagne.csv
Dimensions : (2440, 30)

Apercu :
  code_commune nom_commune code_dept  annee  nb_inscrits  nb_votants  \
0          001    Allineuc        22   2017          419         361   
1          002       Andel        22   2017          858         761   
2          003    Aucaleuc        22   2017          672         580   

   taux_abstention  score_ext_gauche  score_gauche  score_centre  ...  \
0            13.84              2.84         22.38         22.95  ...   
1            11.31              2.43         24.63         33.78  ...   
2            13.69              3.92         32.38         24.73  ...   

   pct_sans_diplome_2016  pct_diplome_sup_2016 pct_sans_diplome_2021  \
